In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/experiment_results.csv")

df["engagement"] = (
    df["likes"]
    + df["replies"]
    + df["reposts"]
    + df["bookmarks"]
)

## success metric

We convert engagement into success/failure.

Use the median engagement.

In [ ]:
threshold = df["engagement"].median()

df["success"] = (df["engagement"] >= threshold).astype(int)

## Count successes per variant

In [ ]:
variants = df["variant"].unique()

success_counts = {}
failure_counts = {}

for v in variants:

    successes = df[(df.variant == v) & (df.success == 1)].shape[0]
    failures = df[(df.variant == v) & (df.success == 0)].shape[0]

    success_counts[v] = successes
    failure_counts[v] = failures

success_counts, failure_counts

## Thompson Sampling function

In [ ]:
def thompson_sampling(success_counts, failure_counts):

    samples = {}

    for variant in success_counts.keys():

        alpha = success_counts[variant] + 1
        beta = failure_counts[variant] + 1

        samples[variant] = np.random.beta(alpha, beta)

    best_variant = max(samples, key=samples.get)

    return best_variant, samples

## Choose the next tweet style

In [ ]:
variant, samples = thompson_sampling(success_counts, failure_counts)

print("Sampled probabilities:", samples)
print("Next tweet variant:", variant)